## Loading the YAML file and extracting the configs

In [0]:
import yaml
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from delta.tables import DeltaTable



CONFIG_PATH = "/Workspace/Users/avranilset@gmail.com/capstone-project/config/config.yaml"

with open(CONFIG_PATH, "r") as f:
    config = yaml.safe_load(f)

CATALOG = config["databricks"]["catalog"]
SILVER_DB = config["databricks"]["schemas"]["silver"]
GOLD_DB = config["databricks"]["schemas"]["gold"]

9

print("GOLD TRANSFORMATION — START")

## Creating the initial dim tables, only needs to be run once

In [0]:
# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_DB}.dim_customer (
#     surrogate_key BIGINT GENERATED ALWAYS AS IDENTITY,
#     customer_id INT,
#     full_name STRING,
#     email STRING,
#     country STRING,
#     signup_date DATE,
#     hash_val STRING,
#     effective_from TIMESTAMP,
#     effective_to TIMESTAMP,
#     is_current BOOLEAN
# )
# """)

# spark.sql(f"""
# CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_DB}.dim_product (
#     surrogate_key BIGINT GENERATED ALWAYS AS IDENTITY,
#     product_id INT,
#     product_name STRING,
#     category STRING,
#     hash_val STRING,
#     effective_from TIMESTAMP,
#     effective_to TIMESTAMP,
#     is_current BOOLEAN
# )
# """)

## SCD2 merge logic function

In [0]:
def scd2_merge(df, table_name, key_col, hash_col):

    df = df.withColumn("effective_from", current_timestamp()) \
           .withColumn("effective_to", lit(None).cast("timestamp")) \
           .withColumn("is_current", lit(True))

    if spark.catalog.tableExists(table_name):

        target = DeltaTable.forName(spark, table_name)

        # STEP 1: Expire old records
        target.alias("t").merge(
            df.alias("s"),
            f"t.{key_col} = s.{key_col} AND t.is_current = true"
        ).whenMatchedUpdate(
            condition=f"t.{hash_col} != s.{hash_col}",
            set={
                "is_current": "false",
                "effective_to": "current_timestamp()"
            }
        ).execute()

        # STEP 2: Insert new + changed
        current_target = spark.table(table_name).filter("is_current = true")

        inserts_df = df.alias("s").join(
            current_target.alias("t"),
            on=key_col,
            how="left"
        ).filter(
            f"t.{key_col} IS NULL OR t.{hash_col} != s.{hash_col}"
        ).select("s.*")

        if inserts_df.limit(1).count() > 0:
            inserts_df.write.format("delta").mode("append").saveAsTable(table_name)

    else:
        df.write.format("delta").mode("overwrite").saveAsTable(table_name)

## Function to build the dimension tables

In [0]:
from pyspark.sql.functions import col, concat, lit, sha2, concat_ws, current_timestamp

def build_dimension_scd2(source_table,target_table,business_key,select_exprs,hash_cols):
    print(f"\n--- Building {target_table.split('.')[-1]} (SCD2) ---")

    df = (
        spark.table(source_table)
        .select(*[expr.alias(col_name) for col_name, expr in select_exprs.items()])
        .withColumn(
            "hash_val",
            sha2(concat_ws("||", *[col(c) for c in hash_cols]), 256)
        )
        .withColumn("effective_from", current_timestamp())
        .withColumn("effective_to", lit(None).cast("timestamp"))
        .withColumn("is_current", lit(True))
    )

    scd2_merge(
        df,
        target_table,
        business_key,
        "hash_val"
    )

## Calling the building function to build the two dimension tables

In [0]:
build_dimension_scd2(
    source_table=f"{CATALOG}.{SILVER_DB}.{config['datasets']['customers']['silver_table']}",
    target_table=f"{CATALOG}.{GOLD_DB}.{config['gold']['dim_customer']}",
    business_key="customer_id",
    select_exprs={
        "customer_id": col("customer_id"),
        "full_name": concat(col("first_name"), lit(" "), col("last_name")),
        "email": col("email"),
        "country": col("country"),
        "signup_date": col("signup_date")
    },
    hash_cols=["full_name", "email", "country", "signup_date"]
)

In [0]:
build_dimension_scd2(
    source_table=f"{CATALOG}.{SILVER_DB}.{config['datasets']['products']['silver_table']}",
    target_table=f"{CATALOG}.{GOLD_DB}.{config['gold']['dim_product']}",
    business_key="product_id",
    select_exprs={
        "product_id": col("product_id"),
        "product_name": col("product_name"),
        "category": col("category")
    },
    hash_cols=["product_name", "category"]
)

For the fact table creation we have not created a seperate function because we are creating only a single fact table

## Building the fact table

In [0]:
print("\n Building fact_sales")

orders = spark.table(f"{CATALOG}.{SILVER_DB}.{config['datasets']['orders']['silver_table']}")

order_items = spark.table(f"{CATALOG}.{SILVER_DB}.{config['datasets']['order_items']['silver_table']}")
products = spark.table(f"{CATALOG}.{SILVER_DB}.{config['datasets']['products']['silver_table']}")
customers = spark.table(f"{CATALOG}.{SILVER_DB}.{config['datasets']['customers']['silver_table']}")

#  LOADING DIMENSIONS 
dim_customer = spark.table(f"{CATALOG}.{GOLD_DB}.{config['gold']['dim_customer']}") \
    .filter("is_current = true") \
    .select("customer_id", col("surrogate_key").alias("customer_sk"))

dim_product = spark.table(f"{CATALOG}.{GOLD_DB}.{config['gold']['dim_product']}") \
    .filter("is_current = true") \
    .select("product_id", col("surrogate_key").alias("product_sk"))

#  FILTERING DELIVERED
delivered = orders.filter(col("status") == config["gold"]["fact_filter_status"])

#  JOINING USING SURROGATE KEYS
joined = delivered.join(order_items, "order_id") \
    .join(products.select("product_id", "price"), "product_id") \
    .join(broadcast(dim_product), "product_id") \
    .join(dim_customer, "customer_id")

# BUILDING FACT TABLE
df = (
    joined
    .withColumn(
        "sale_id",
        sha2(
            concat_ws("||",
                col("customer_sk"),
                col("order_id"),
                col("order_item_id"),
                col("product_sk")
            ),
            256
        )
    )
    .withColumn("total_amount", round(col("quantity") * col("price"), 2))
    .select(
        "sale_id",
        "order_id",
        "order_item_id",
        "customer_sk",
        "product_sk",
        col("quantity"),
        col("price").alias("unit_price"),
        "total_amount",
        "order_date",
        "status",
        year("order_date").alias("order_year"),
        month("order_date").alias("order_month"),
    )
)
fact_table = f"{CATALOG}.{GOLD_DB}.{config['gold']['fact_sales']}"

if spark.catalog.tableExists(fact_table):

    target = DeltaTable.forName(spark, fact_table)

    target.alias("t").merge(
        df.alias("s"),
        "t.sale_id = s.sale_id"
    ).whenMatchedUpdateAll() \
     .whenNotMatchedInsertAll() \
     .whenNotMatchedBySourceDelete() \
     .execute()

else:
    df.write.format("delta") \
        .mode("overwrite") \
        .partitionBy(config["gold"]["partition_column"]) \
        .saveAsTable(fact_table)

## Optimizations

In [0]:
spark.sql(f"OPTIMIZE {fact_table} ZORDER BY (customer_sk, product_sk)")
spark.sql(f"OPTIMIZE {CATALOG}.{GOLD_DB}.{config['gold']['dim_customer']}")
spark.sql(f"OPTIMIZE {CATALOG}.{GOLD_DB}.{config['gold']['dim_product']}")

print("GOLD TRANSFORMATION — COMPLETE")